In [ ]:
import cv2
import numpy as np

In [ ]:
PATH2DATA = 'C://Users/abram/Downloads/angio-6/slice.png'

Load image

In [ ]:
def get_img():
    img = cv2.imread(PATH2DATA + '/train/Original/2_A.png')
    return cv2.resize(img, (512, 512))

def get_anio():
    img = cv2.imread("C://Users/abram/Downloads/angio-6/slice.png",cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img,(512,512))
    cv2.imshow('orig',img)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(15,15))
    img = clahe.apply(img)
    img = cv2.GaussianBlur(img,(7,7),1.4)
    #edges = cv2.Canny(img,30,105)
    cv2.imshow('anio',img)
    cv2.waitKey(0)

def get_green():
    img = get_img()
    green = img.copy()
    green[:,:,0] = 0
    green[:,:,2] = 0
    return green

get_anio()

Green channel extraction

In [ ]:
img2draw = get_img()
green = img2draw.copy()
green[:,:,0] = 0
green[:,:,2] = 0
cv2.imshow('green', green)
cv2.waitKey()

Porposed method

In [ ]:
green = get_green()
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(15,15))
lab = cv2.cvtColor(green,cv2.COLOR_BGR2Lab)
lab_planes = list(cv2.split(lab))
lab_planes[0] = clahe.apply(lab_planes[0])
lab = cv2.merge(lab_planes)
green_clahe = cv2.cvtColor(lab,cv2.COLOR_Lab2BGR)
# cv2.imshow('green_clahe',green_clahe)
# cv2.imshow('green',green)
# cv2.waitKey(0)

In [ ]:
gray_green = cv2.cvtColor(green_clahe,cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray_green, (7, 7), 1.4, 1.4)
cv2.imshow('gg',blurred)
cv2.waitKey(0)

In [ ]:
blurred = cv2.GaussianBlur(green_clahe, (5, 5), 1.4, 1.4)
gx = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3)  # Градиент по оси X
gy = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3)  # Градиент по оси Y
M = cv2.magnitude(gx, gy)
angle = np.atan(gy/gx)


In [ ]:
img2draw = get_img()
gray = cv2.cvtColor(img2draw, cv2.COLOR_BGR2GRAY)
cv2.imshow('Original', gray)
cv2.waitKey(0)

In [ ]:
thresh = cv2.adaptiveThreshold(green_clahe[:,:,1], 250, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,53,10)
cv2.imwrite('adaptive_thresh.png',thresh)
cv2.waitKey(0)

Adaptive threshold

In [ ]:
img2draw = get_img()
thresh = cv2.adaptiveThreshold(green_clahe[:,:,1], 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,53,10)
contours, hierarchy = cv2.findContours(thresh, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(img2draw, contours, -1,(255,0,0),2)
cv2.imshow('test',img2draw)
cv2.waitKey(0)

Canny

In [ ]:
img2draw = get_img()
blurred = cv2.GaussianBlur(green, (5, 5), 0)
#thresh = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,43,13)
edges = cv2.Canny(blurred,25,55)
contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(img2draw, contours, -1,(255,0,0),2)
cv2.imshow('canny',edges)
cv2.waitKey(0)

In [ ]:
img2draw = get_img()
edges = cv2.Canny(img,100,200)
contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(img2draw, contours, -1,(255,0,0),2)
cv2.imshow('canny',edges)
cv2.waitKey(0)

Sobel

In [ ]:

sobel_combined = np.uint8(np.absolute(sobel_combined))
img2draw = img.copy()
thresh = cv2.adaptiveThreshold(sobel_combined,255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,53,10)
contours, _ = cv2.findContours(thresh, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(img2draw, contours, -1,(255,0,0),2)
cv2.imshow('sobel',img2draw)
cv2.waitKey(0)

In [ ]:
import cv2

img = cv2.imread("C:\\Users\\abram\\Downloads\\test (3).png", cv2.IMREAD_GRAYSCALE)
canny = cv2.Canny(img,127,255)
adap = cv2.adaptiveThreshold(img,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY,15,3)
_, thresh = cv2.threshold(img,127,255,cv2.THRESH_BINARY)
cv2.imshow('img', img)
cv2.imshow('adaptive', adap)
cv2.imshow('canny', canny)
cv2.imshow('thresh', thresh)
cv2.waitKey(0)

ITK testing

In [ ]:
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt
import SimpleITK as sitk

img = sitk.ReadImage("C:\\Users\\abram\\practice\\VCD\\angio-6\\test_img.png")



In [ ]:
def apply_conn_thresh(file_path, blur, low_threshold, high_threshold):
    img = sitk.ReadImage(file_path)
    img = sitk.BinomialBlur(img, blur)
    conn_thresh = sitk.ConnectedThreshold(img, seedList=[(132, 142, 96)], lower=low_threshold, upper=high_threshold)
    plt.figure(figsize=(12, 6))
    plt.imshow(sitk.GetArrayViewFromImage(conn_thresh), cmap='gray')
    plt.title(f'Low={low_threshold}, High={high_threshold}')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

interact(apply_conn_thresh,
    file_path = 
    low_threshold=IntSlider(min=0, max=255, step=1, value=50, description='Low Threshold'),
    high_threshold=IntSlider(min=0, max=255, step=1, value=150, description='High Threshold')) #72 149

interactive(children=(IntSlider(value=50, description='Low Threshold', max=255), IntSlider(value=150, descript…

<function __main__.apply_conn_thresh(low_threshold, high_threshold)>

OTSU

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

def otsu_segmentation(image_path):
    # Загрузка изображения в оттенках серого
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError("Изображение не загружено. Проверьте путь.")
    
    # Вычисление гистограммы
    hist = cv2.calcHist([img], [0], None, [256], [0, 256])
    
    # Функция для интерактивного отображения
    def apply_otsu(blur_size=5, threshold_adjust=0):
        # Применение размытия (опционально)
        if blur_size > 0:
            blurred = cv2.GaussianBlur(img, (blur_size, blur_size), 0)
        else:
            blurred = img
        
        # Автоматический порог по Отсу
        ret, otsu_thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Корректировка порога (если нужно)
        adjusted_thresh = ret + threshold_adjust
        _, adjusted_thresh_img = cv2.threshold(blurred, adjusted_thresh, 255, cv2.THRESH_BINARY)
        
        # Отображение результатов
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        plt.imshow(blurred, cmap='gray')
        plt.title(f'Исходное (размытие {blur_size}x{blur_size})')
        plt.axis('off')
        
        plt.subplot(1, 3, 2)
        plt.imshow(otsu_thresh, cmap='gray')
        plt.title(f'Otsu threshold: {ret:.1f}')
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        plt.imshow(adjusted_thresh_img, cmap='gray')
        plt.title(f'Adjusted threshold: {adjusted_thresh:.1f}')
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
    
    # Создание интерактивных слайдеров
    interact(
        apply_otsu,
        blur_size=IntSlider(min=0, max=20, step=1, value=5, description='Размытие:'),
        threshold_adjust=IntSlider(min=-70, max=50, step=1, value=0, description='Корректировка:')
    )

# Укажите путь к вашему изображению
image_path = 'C:\\Users\\abram\\practice\\VCD\\angio-6\\test_img.png'  # Замените на ваш путь
otsu_segmentation(image_path)

interactive(children=(IntSlider(value=5, description='Размытие:', max=20), IntSlider(value=0, description='Кор…

Fast Marching

In [42]:
def ffm(inputFilename, seedX, seedY, sigma, alpha, beta, timeThreshold, stoppingTime):
    seedPosition = (seedX, seedY)
    inputImage = sitk.ReadImage(inputFilename, sitk.sitkFloat32)
    smoothing = sitk.CurvatureAnisotropicDiffusionImageFilter()
    smoothing.SetTimeStep(0.125)
    smoothing.SetNumberOfIterations(5)
    smoothing.SetConductanceParameter(9.0)
    smoothingOutput = smoothing.Execute(inputImage)

    gradientMagnitude = sitk.GradientMagnitudeRecursiveGaussianImageFilter()
    gradientMagnitude.SetSigma(sigma)
    gradientMagnitudeOutput = gradientMagnitude.Execute(smoothingOutput)

    sigmoid = sitk.SigmoidImageFilter()
    sigmoid.SetOutputMinimum(0.0)
    sigmoid.SetOutputMaximum(1.0)
    sigmoid.SetAlpha(alpha)
    sigmoid.SetBeta(beta)
    sigmoidOutput = sigmoid.Execute(gradientMagnitudeOutput)

    fastMarching = sitk.FastMarchingImageFilter()

    seedValue = 0
    trialPoint = (seedPosition[0], seedPosition[1], seedValue)

    fastMarching.AddTrialPoint(trialPoint)

    fastMarching.SetStoppingValue(stoppingTime)

    fastMarchingOutput = fastMarching.Execute(sigmoidOutput)

    thresholder = sitk.BinaryThresholdImageFilter()
    thresholder.SetLowerThreshold(0.0)
    thresholder.SetUpperThreshold(timeThreshold)
    thresholder.SetOutsideValue(0)
    thresholder.SetInsideValue(255)

    result = thresholder.Execute(fastMarchingOutput)

    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.title('Input Image')
    plt.imshow(sitk.GetArrayViewFromImage(inputImage), cmap='gray')
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.title('Result Image')
    plt.imshow(sitk.GetArrayViewFromImage(result), cmap='gray')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    

In [44]:
%matplotlib inline
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider


interact(
    ffm,
    inputFilename='C:\\Users\\abram\\practice\\VCD\\angio-6\\test_img.png',
    seedX=IntSlider(min=0, max=512, step=1, value=178, description='seedx'),
    seedY=IntSlider(min=0, max=512, step=1, value=232, description='seedy'),
    sigma=FloatSlider(min=0.1, max=10.0, step=0.1, value=1.0),
    alpha=FloatSlider(min=0.1, max=10.0, step=0.1, value=0.5),
    beta=FloatSlider(min=0.1, max=10.0, step=0.1, value=0.5),
    timeThreshold=FloatSlider(min=0.1, max=1000.0, step=0.1, value=100.0),
    stoppingTime=FloatSlider(min=0.1, max=1000.0, step=0.1, value=100.0)
)

interactive(children=(Text(value='C:\\Users\\abram\\practice\\VCD\\angio-6\\test_img.png', description='inputF…

<function __main__.ffm(inputFilename, seedX, seedY, sigma, alpha, beta, timeThreshold, stoppingTime)>

adaptive

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

def adaptive_threshold_segmentation(image_path):
    # Загрузка изображения в оттенках серого
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError("Изображение не загружено. Проверьте путь.")
    
    # Функция для интерактивного отображения
    def apply_adaptive(blur_size=5, block_size=11, C=2, method='mean'):
        # Применение размытия
        if blur_size > 0:
            blurred = cv2.GaussianBlur(img, (blur_size, blur_size), 0)
        else:
            blurred = img
        
        # Корректировка block_size (должен быть нечетным)
        block_size = max(3, block_size | 1)  # Обеспечиваем нечетность
        
        # Выбор метода адаптивного порога
        if method == 'gaussian':
            adaptive_method = cv2.ADAPTIVE_THRESH_GAUSSIAN_C
        else:
            adaptive_method = cv2.ADAPTIVE_THRESH_MEAN_C
        
        # Применение адаптивного порога
        binary = cv2.adaptiveThreshold(
            blurred, 
            255, 
            adaptive_method, 
            cv2.THRESH_BINARY, 
            block_size, 
            C
        )
        
        # Отображение результатов
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 2, 1)
        plt.imshow(blurred, cmap='gray')
        plt.title(f'Исходное (размытие {blur_size}x{blur_size})')
        plt.axis('off')
        
        plt.subplot(1, 2, 2)
        plt.imshow(binary, cmap='gray')
        plt.title(f'Adaptive: {method}\nBlock Size: {block_size}, C: {C}')
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
    
    # Создание интерактивных слайдеров
    interact(
        apply_adaptive,
        blur_size=IntSlider(min=0, max=20, step=1, value=5, description='Размытие:'),
        block_size=IntSlider(min=3, max=55, step=2, value=11, description='Размер блока:'),
        C=IntSlider(min=-10, max=20, step=1, value=2, description='Константа C:'),
        method=['mean', 'gaussian']
    )

# Укажите путь к вашему изображению
image_path = 'C:\\Users\\abram\\practice\\VCD\\angio-6\\test_img.png'  # Замените на ваш путь
adaptive_threshold_segmentation(image_path)

interactive(children=(IntSlider(value=5, description='Размытие:', max=20), IntSlider(value=11, description='Ра…